# Checkpoint 10 — WAPE-first XGBoost selection

This checkpoint keeps the four horizons independent, uses **WAPE as the primary metric**, and does not report MAPE. Title scores and thumbnail features remain optional future inputs. The reserved test partition is not used here.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint10_wape_blended'
HORIZONS = (7, 14, 21, 30)
print('Project:', PROJECT_ROOT)
print('Artifact:', ARTIFACT_DIR)

## 1. Build or load the checkpoint

Set `RUN_TRAINING = True` to retrain the final component models. The expensive objective comparison is already persisted by `scripts/compare_wape_objectives.py`.

In [ ]:
RUN_TRAINING = False
required = [
    ARTIFACT_DIR / 'selected_ensembles.csv',
    ARTIFACT_DIR / 'cross_fitted_blend_predictions.csv',
    ARTIFACT_DIR / 'validation_tests.csv',
    ARTIFACT_DIR / 'training_manifest.json',
]
if RUN_TRAINING or not all(path.exists() for path in required):
    from scripts.build_wape_blend import build_blended_models
    build_blended_models(project_root=PROJECT_ROOT)
else:
    print('PASS — persisted WAPE checkpoint found; no retraining needed.')

## 2. What ‘unseen channels across folds’ means

For a validation fold, **all videos from a channel are held out together**. A channel in validation therefore has zero videos in that fold's training rows. This prevents the model from learning a channel from its other videos and gives an honest new-channel estimate. `channel_id` is used only to create the split; it is not a model feature. Subscriber count, subscriber tier and the clean historical average can still be model inputs.

In [ ]:
split_checks = []
for horizon in HORIZONS:
    path = PROJECT_ROOT / 'Dataset' / 'model_split_metadata' / f'viewcastlk_day_{horizon}_split_assignments.csv'
    assignments = pd.read_csv(path, low_memory=False)
    development = assignments['partition'].eq('development')
    for fold in range(1, 6):
        validation = development & assignments['cv_validation_fold'].eq(fold)
        training = development & ~validation
        train_channels = set(assignments.loc[training, 'channel_id'].astype(str))
        validation_channels = set(assignments.loc[validation, 'channel_id'].astype(str))
        overlap = train_channels & validation_channels
        split_checks.append({
            'horizon': f'Day {horizon}', 'fold': fold,
            'training_channels': len(train_channels),
            'validation_channels': len(validation_channels),
            'overlapping_channels': len(overlap),
            'result': 'PASS' if not overlap else 'FAIL',
        })
split_checks = pd.DataFrame(split_checks)
display(split_checks)
assert split_checks['overlapping_channels'].eq(0).all()
print('PASS — every validation channel is unseen in that fold training set.')

## 3. Why WAPE matches the stated error preference

WAPE is `sum(abs(actual - predicted)) / sum(actual)`. The larger miss supplies almost all of the error numerator below, even though no per-row percentage is used. Lower is better; 0% is perfect.

In [ ]:
metric_example = pd.DataFrame({
    'case': ['very low-view miss', 'important 32K-view miss'],
    'actual_views': [2, 32_295],
    'predicted_views': [28, 2_798],
})
metric_example['absolute_error_views'] = (metric_example['actual_views'] - metric_example['predicted_views']).abs()
metric_example['share_of_total_absolute_error_pct'] = 100 * metric_example['absolute_error_views'] / metric_example['absolute_error_views'].sum()
display(metric_example.round(3))
example_wape = 100 * metric_example['absolute_error_views'].sum() / metric_example['actual_views'].sum()
print(f'Combined WAPE: {example_wape:.2f}%')
print(f'The 32K miss contributes {metric_example.loc[1, "absolute_error_views"] / metric_example.loc[0, "absolute_error_views"]:,.0f} times more error than the 2-to-28 miss.')

## 4. Objective and blend selection

Each candidate was evaluated on the same five saved development folds. A blend weight applied to one fold was learned from the other four folds, so that fold did not choose its own weight. Here are the best five choices for every horizon.

In [ ]:
blend_summary = pd.read_csv(ARTIFACT_DIR / 'cross_fitted_blend_summary.csv')
best_five = (blend_summary.sort_values(['horizon_days', 'wape_pct', 'rmsle'])
             .groupby('horizon_days', as_index=False).head(5))
display(best_five[[
    'horizon_days', 'candidate_a', 'candidate_b',
    'fold_weights_on_candidate_a', 'wape_pct',
    'total_view_capture_pct', 'top_decile_view_capture_pct', 'rmsle'
]].round(3))

## 5. Selected model for each independent horizon

`log_squared_*` models stabilize the wide view distribution; `raw_absolute` is more sensitive to view-scale error. Where useful, the final candidate averages their predictions using the shown weights.

In [ ]:
selected = pd.read_csv(ARTIFACT_DIR / 'selected_ensembles.csv')
display(selected[[
    'horizon_days', 'components', 'weights',
    'cross_fitted_oof_wape_pct',
    'cross_fitted_oof_total_view_capture_pct',
    'cross_fitted_oof_top_decile_view_capture_pct',
    'cross_fitted_oof_rmsle'
]].round(3))

In [ ]:
manifest = json.loads((ARTIFACT_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
combined = pd.DataFrame([manifest['selected_oof_combined_metrics']])
display(combined.round(3))
assert manifest['primary_metric'] == 'wape_pct'
assert manifest['mape_status'] == 'not_reported_or_used'
assert manifest['reserved_test_used'] is False
print('PASS — WAPE is primary, MAPE is absent, and the reserved test remains untouched.')

### Current limitation visible in the metrics

Overall view capture is about 22%, and top-decile capture is about 11%. So this checkpoint still underpredicts viral/high-view videos badly. WAPE-first training improved the earlier candidate only modestly; it did not manufacture predictive information that is absent from the current features. Title scores and thumbnail features can plug into the existing preprocessing contract later.

## 6. Inspect real out-of-fold predictions

These are development-fold predictions, not training-row fitted values and not reserved-test predictions. Rows near several actual-view quantiles are shown for each horizon.

In [ ]:
oof = pd.read_csv(ARTIFACT_DIR / 'cross_fitted_blend_predictions.csv', low_memory=False)
example_rows = []
for horizon, group in oof.groupby('horizon_days'):
    ordered = group.sort_values('actual_views').reset_index(drop=True)
    positions = np.unique(np.rint(np.linspace(0, len(ordered) - 1, 8)).astype(int))
    sample = ordered.iloc[positions][['horizon_days', 'video_id', 'actual_views', 'predicted_views']].copy()
    sample['absolute_error_views'] = (sample['actual_views'] - sample['predicted_views']).abs()
    example_rows.append(sample)
display(pd.concat(example_rows, ignore_index=True).round({'predicted_views': 1, 'absolute_error_views': 1}))

## 7. Automated checkpoint tests

In [ ]:
tests = pd.read_csv(ARTIFACT_DIR / 'validation_tests.csv')
display(tests)
assert tests['status'].eq('PASS').all()
assert len(selected) == 4 and set(selected['horizon_days']) == set(HORIZONS)
print(f'PASS — all {len(tests)} artifact checks passed and all four horizons have saved models.')